# Compte rendu de TP – Spark 

In [1]:
import os, subprocess
os.environ["JAVA_HOME"] = subprocess.check_output(
    ["/usr/libexec/java_home", "-v", "17"]).decode().strip()

In [2]:
import pandas as pd 
from datetime import time
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StringType


## 1. Optimisation Spark et Spark UI

In [3]:
# Session spark et chargement des datasets 
spark = SparkSession.builder.appName("TP1").master("local[*]").getOrCreate()
sales_big_df = spark.read.csv("sales_big.csv", header=True, inferSchema=True)
sales_df = spark.read.csv("sales.csv", header=True, inferSchema=True)
regions_df = spark.read.csv("regions.csv", header=True, inferSchema=True)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/24 10:08:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
sales_big_df.head(5)

[Row(id=1, date=datetime.date(2024, 3, 6), produit='Imprimante', categorie='Informatique', quantite=14, prix_unitaire=150.0, region='Nord'),
 Row(id=2, date=datetime.date(2024, 10, 14), produit='Chargeur', categorie='Accessoires', quantite=15, prix_unitaire=15.0, region='Ouest'),
 Row(id=3, date=datetime.date(2024, 2, 9), produit='Souris', categorie='Accessoires', quantite=7, prix_unitaire=25.5, region='Ouest'),
 Row(id=4, date=datetime.date(2024, 11, 29), produit='Disque SSD', categorie='Informatique', quantite=2, prix_unitaire=120.0, region='Nord'),
 Row(id=5, date=datetime.date(2024, 1, 29), produit='Imprimante', categorie='Informatique', quantite=7, prix_unitaire=150.0, region='Sud')]

### Temps d'éxécution avec / sans cache

In [5]:
spark.catalog.clearCache()

#### Aggrégation sans cache

In [6]:
%%time 
sales_big_df.groupBy("region", "categorie").agg(F.sum("quantite").alias("total_vendu")).show()

+------+------------+-----------+
|region|   categorie|total_vendu|
+------+------------+-----------+
|  Nord| Accessoires|     130116|
|   Sud|Informatique|     129774|
| Ouest|Informatique|     130326|
|  Nord|Informatique|     132866|
|   Sud| Accessoires|     130839|
|   Est|Informatique|     133457|
|   Est| Accessoires|     128514|
| Ouest| Accessoires|     133995|
+------+------------+-----------+

CPU times: user 1.38 ms, sys: 857 μs, total: 2.24 ms
Wall time: 654 ms


#### Aggrégation avec cache

In [7]:
# Mise en cache
sales_big_cache = sales_big_df.cache()
sales_big_cache.count()

100000

In [8]:
%%time 
sales_big_cache.groupBy("region", "categorie").agg(F.sum("quantite").alias("total_vendu")).show()

+------+------------+-----------+
|region|   categorie|total_vendu|
+------+------------+-----------+
|  Nord| Accessoires|     130116|
|   Sud|Informatique|     129774|
| Ouest|Informatique|     130326|
|  Nord|Informatique|     132866|
|   Sud| Accessoires|     130839|
|   Est|Informatique|     133457|
|   Est| Accessoires|     128514|
| Ouest| Accessoires|     133995|
+------+------------+-----------+

CPU times: user 2.98 ms, sys: 2.86 ms, total: 5.84 ms
Wall time: 156 ms


#### Résultats : agrégation avec vs sans cache

| Configuration | Wall time |
|---|---|
| Sans cache | 349 ms |
| Avec cache | 192 ms |
| **Gain** | **1,8× plus rapide** |

**Sans cache**, Spark repart du fichier à chaque action : il relit `sales_big.csv` sur le disque, découpe chaque ligne et convertit les colonnes dans leur type (`int`, `double`, `date`). Ce n'est qu'ensuite qu'il peut faire l'agrégation.

**Avec cache**, ce travail n'est fait qu'une seule fois. Le premier `count()` stocke le DataFrame déjà lu et converti en mémoire. Les agrégations suivantes partent directement de cette copie et sautent toute l'étape de lecture.


### Partitions

In [9]:
# Nombre initial de partitions (par défaut)
print("Partitions initialses:", sales_big_df.rdd.getNumPartitions())

Partitions initialses: 2


Le CSV est initialement lu en 2 partitions. On le redistribue en 8 partitions (`df8`), qu'on met en cache. On passe ensuite de 8 à 2 partitions de deux façons différentes, et on lance un `count()` pour déclencher le calcul.

In [10]:
# Nombre de partitions de départ à 8 
df8 = sales_big_df.repartition(8).cache()
df8.count()
print("Partitions :", df8.rdd.getNumPartitions())

Partitions : 8


#### Avec repartition(2)


In [11]:
%%time
spark.sparkContext.setJobDescription("repartition(2)")
df_rep = df8.repartition(2)
df_rep.count()

CPU times: user 1.2 ms, sys: 1.84 ms, total: 3.04 ms
Wall time: 181 ms


100000

#### Avec coalesce(2)

In [12]:
%%time
spark.sparkContext.setJobDescription("coalesce(2)")
df_coal = df8.coalesce(2)
df_coal.count()

CPU times: user 863 μs, sys: 1.28 ms, total: 2.14 ms
Wall time: 66.3 ms


100000

In [13]:
print("repartition(2) :", df_rep.rdd.glom().map(len).collect())
print("coalesce(2)    :", df_coal.rdd.glom().map(len).collect())


repartition(2) : [50000, 50000]
coalesce(2)    : [50001, 49999]


In [14]:
df_rep.explain()
df_coal.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=true
+- == Final Plan ==
   ResultQueryStage 2
   +- ShuffleQueryStage 1
      +- Exchange RoundRobinPartitioning(2), REPARTITION_BY_NUM, [plan_id=435]
         +- TableCacheQueryStage 0
            +- InMemoryTableScan [id#17, date#18, produit#19, categorie#20, quantite#21, prix_unitaire#22, region#23]
                  +- InMemoryRelation [id#17, date#18, produit#19, categorie#20, quantite#21, prix_unitaire#22, region#23], StorageLevel(disk, memory, deserialized, 1 replicas)
                        +- AdaptiveSparkPlan isFinalPlan=true
                        +- == Final Plan ==
                           ResultQueryStage 2
                           +- ShuffleQueryStage 1
                              +- Exchange RoundRobinPartitioning(8), REPARTITION_BY_NUM, [plan_id=251]
                                 +- TableCacheQueryStage 0
                                    +- InMemoryTableScan [id#17, date#18, produit#19, categorie#20, quan

#### Résultats : repartition vs coalesce

| | `repartition(2)` | `coalesce(2)` |
|---|---|---|
| Wall time | 241 ms | 193 ms |
| Lignes par partition | `[50000, 50000]` | `[50001, 49999]` |
| Opération | `Exchange RoundRobinPartitioning(2)` | `Coalesce 2` |
| Nombre de stages | 2 (`ResultQueryStage 2`) | 1 (`ResultQueryStage 1`) |


- Les deux méthodes donnent bien 2 partitions et le même nombre total de lignes (100 000).
- coalesce(2) est environ 50 ms plus rapide que repartition(2).
- repartition produit des partitions parfaitement égales. Celles de coalesce sont presque égales, car les 8 partitions de départ étaient déjà équilibrées.
- Le plan de repartition contient une étape Exchange (un shuffle), ce qui donne 2 stages. Celui de coalesce n'en contient pas et s'exécute en 1 seul stage.


### Jointure avec broadcast()

On joint une grosse table (big_sales_df) avec une petite table (regions_df) sur la colonne region, avec puis sans broadcast.

In [15]:
# Désactiver broadcast auto
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

#### Jointure sans broadcast 

In [16]:
%%time
spark.sparkContext.setJobDescription("jointure sans broadcast")
join_no_broadcast = sales_big_cache.join(regions_df, on="region")
join_no_broadcast.count()

CPU times: user 1.48 ms, sys: 1.91 ms, total: 3.38 ms
Wall time: 546 ms


100000

#### Jointure avec broadcast

In [17]:
%%time
spark.sparkContext.setJobDescription("jointure avec broadcast")
join_broadcast = sales_big_cache.join(F.broadcast(regions_df), on="region")
join_broadcast.count()

CPU times: user 1.6 ms, sys: 2.81 ms, total: 4.41 ms
Wall time: 195 ms


100000

In [18]:
join_no_broadcast.explain()
join_broadcast.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [region#23, id#17, date#18, produit#19, categorie#20, quantite#21, prix_unitaire#22, chef_lieu#66, pays#67]
   +- SortMergeJoin [region#23], [region#65], Inner
      :- Sort [region#23 ASC NULLS FIRST], false, 0
      :  +- Exchange hashpartitioning(region#23, 200), ENSURE_REQUIREMENTS, [plan_id=860]
      :     +- Filter isnotnull(region#23)
      :        +- InMemoryTableScan [id#17, date#18, produit#19, categorie#20, quantite#21, prix_unitaire#22, region#23], [isnotnull(region#23)]
      :              +- InMemoryRelation [id#17, date#18, produit#19, categorie#20, quantite#21, prix_unitaire#22, region#23], StorageLevel(disk, memory, deserialized, 1 replicas)
      :                    +- FileScan csv [id#17,date#18,produit#19,categorie#20,quantite#21,prix_unitaire#22,region#23] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/Users/antoinechosson/Spark-lab/Spark-Introduction-La

In [19]:
# Rétablir le réglage par défaut
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10 * 1024 * 1024)

#### Résultats : jointure avec vs sans broadcast


| | Sans broadcast | Avec broadcast |
|---|---|---|
| Wall time | 483 ms | 174 ms |
| Type de jointure | SortMergeJoin | BroadcastHashJoin |
| Déplacement de sales_big | shuffle par région | aucun |
| Déplacement de regions | shuffle par région | copie envoyée à chaque partition |
| Tri avant jointure | sur les deux tables | aucun |

La jointure avec broadcast est environ 2,8 fois plus rapide.

Sans broadcast (SortMergeJoin) :
- Spark doit rapprocher les lignes qui ont la même région. Il redistribue donc les deux tables par région avec un shuffle des deux côtés.
- Il trie ensuite chaque côté par région, puis fusionne les deux listes triées.
- Les données sont réparties en 200 partitions (valeur par défaut de Spark). Il n'y a pourtant que 4 régions, donc presque toutes ces partitions sont vides : Spark lance 200 tâches pour rien.

Avec broadcast (BroadcastHashJoin) :
- Spark envoie une copie complète de regions à chaque partition de sales_big.
- Chaque partition fait sa jointure localement, en cherchant la région de chaque ligne dans cette petite table.
- Pas de shuffle et pas de tri sur la grosse table.

### Analyse sur SparkUI

In [20]:
spark.sparkContext.uiWebUrl

'http://macbookpro.home:4040'

Dans la Spark UI, chaque action est découpée en plusieurs jobs, un par étape. Les stages marqués "skipped" sont des étapes déjà calculées (cache ou shuffle précédent) que Spark réutilise sans les refaire.

| Opération | Stages exécutés | Tâches par stage | Shuffles |
|---|---|---|---|
| repartition(2) | 3 | 8 → 2 → 1 | 2 |
| coalesce(2) | 2 | 2 → 1 | 1 |
| Jointure sans broadcast | 4 | 2, 1 → 1 → 1 | 3 |
| Jointure avec broadcast | 3 | 1, 2 → 1 | 1 |

Le dernier stage à 1 tâche vient du count() : chaque partition compte ses lignes, puis Spark rassemble les sous-totaux. Il est présent dans tous les tests.

- Le nombre de tâches d'un stage correspond au nombre de partitions traitées : 8 tâches quand on lit df8, 2 tâches après le passage à 2 partitions.
- repartition(2) ajoute un stage et un shuffle par rapport à coalesce(2)
- La jointure sans broadcast nécessite deux shuffles : un pour sales_big et un pour regions, afin de regrouper les lignes de même région.
- Avec broadcast, regions est lue une fois puis copiée vers les partitions de sales_big.
- Le plan prévoyait 200 partitions pour la jointure sans broadcast, mais Spark les a fusionnées en une seule pendant l'exécution, car les données étaient très petites.


### Pourquoi repartition déclenche-t-il un shuffle alors que coalesce non ?

repartition(n) redistribue toutes les lignes dans n nouvelles partitions, à tour de rôle ou selon une clé. N'importe quelle ligne peut donc changer de partition. Spark doit écrire toutes les données, les échanger entre les partitions puis les relire : c'est un shuffle. En contrepartie, les partitions obtenues sont équilibrées, et repartition peut aussi bien augmenter que diminuer leur nombre.

coalesce(n) ne redistribue rien. Il regroupe des partitions existantes entre elles : chaque nouvelle partition lit simplement plusieurs anciennes partitions, telles qu'elles sont. Aucune ligne n'est déplacée individuellement, il n'y a donc pas de shuffle. En contrepartie, coalesce ne peut que réduire le nombre de partitions, et leur taille n'est pas garantie équilibrée.

## 2. DatFrames avancés et Window Functions

In [21]:
spark = SparkSession.builder.appName("TP2").master("local[*]").getOrCreate()
customers_df = spark.read.csv("/Users/antoinechosson/Spark-lab/Spark-Introduction-Lab/tp2/customers.csv", header=True, inferSchema=True)
sales_df = spark.read.csv("/Users/antoinechosson/Spark-lab/Spark-Introduction-Lab/tp2/sales.csv", header=True, inferSchema=True)

26/10/24 10:08:29 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [22]:
# Jointure des deux tables sur customer_id
df = sales_df.join(F.broadcast(customers_df), on="customer_id")
df.show(3)

+-----------+---+----------+-------+-----------+--------+-------------+------+-------+-------+-----+--------+----------------+
|customer_id| id|      date|produit|  categorie|quantite|prix_unitaire|region|    nom| prenom|ville| segment|date_inscription|
+-----------+---+----------+-------+-----------+--------+-------------+------+-------+-------+-----+--------+----------------+
|         15|  1|2024-01-04| Souris|Accessoires|       5|         25.5|   Sud|Bernard|  Manon|Lille| Premium|      2022-02-11|
|         38|  2|2024-01-14|Clavier|Accessoires|       9|         45.0|  Nord| Durand|   Emma|Lille|Standard|      2021-05-31|
|         14|  3|2024-01-05| Casque|Accessoires|       1|         35.0|  Nord| Moreau|Nicolas| Lyon|     VIP|      2022-01-29|
+-----------+---+----------+-------+-----------+--------+-------------+------+-------+-------+-----+--------+----------------+
only showing top 3 rows


### CA cumulé par client 

In [23]:
# Ajout du montant de chaque vente
df = df.withColumn("montant", F.col("quantite")*F.col("prix_unitaire"))
# Fenêtre de calcul 
W = (Window.partitionBy('customer_id')
          .orderBy("date", "id")
          .rowsBetween(Window.unboundedPreceding, Window.currentRow))

ca_cumule_df = df.withColumn("ca_cumule", F.sum("montant").over(W))
ca_cumule_df[["customer_id", "date", "ca_cumule"]].show(3)

+-----------+----------+---------+
|customer_id|      date|ca_cumule|
+-----------+----------+---------+
|          1|2024-01-03|   1050.0|
|          1|2024-01-09|   1590.0|
|          1|2024-01-13|   4290.0|
+-----------+----------+---------+
only showing top 3 rows


### Classement des produits par CA au sein de chaque catégorie

In [24]:
ca_produit_df = (df.groupBy("categorie", "produit")
                 .agg(F.sum("montant").alias("ca")))

rank_window = Window.partitionBy("categorie").orderBy(F.desc("ca"))
classement_ca_df = ca_produit_df.withColumn("rang", F.rank().over(rank_window))
classement_ca_df.orderBy("categorie", "rang").show()


+------------+----------+--------+----+
|   categorie|   produit|      ca|rang|
+------------+----------+--------+----+
| Accessoires|    Webcam| 16920.0|   1|
| Accessoires|   Clavier| 11475.0|   2|
| Accessoires|    Casque| 10115.0|   3|
| Accessoires|    Souris|  8619.0|   4|
| Accessoires|  Chargeur|  5355.0|   5|
|Informatique|Ordinateur|219300.0|   1|
|Informatique|  Tablette|109800.0|   2|
|Informatique|     Écran| 67320.0|   3|
|Informatique|Imprimante| 36450.0|   4|
|Informatique|Disque SSD| 21960.0|   5|
+------------+----------+--------+----+



### Catégorisation des ventes

Choix des catégories en fonction des quantiles p33 et p66

In [25]:
df.agg(F.percentile_approx("montant", 0.33)).show()
df.agg(F.percentile_approx("montant", 0.66)).show()

+---------------------------------------+
|percentile_approx(montant, 0.33, 10000)|
+---------------------------------------+
|                                  210.0|
+---------------------------------------+

+---------------------------------------+
|percentile_approx(montant, 0.66, 10000)|
+---------------------------------------+
|                                  750.0|
+---------------------------------------+



#### Catégorisation avec UDF

In [26]:
def classify_sale(amount):  
    if amount is None: 
        return None
    if amount <= 210: 
        return "petit"
    elif amount > 210 and amount < 750: 
        return "moyen"
    else: 
        return "gros"


classify_sale_udf = F.udf(classify_sale, StringType())


In [27]:
%%time
categorized_df_UDF = df.withColumn("taille_vente", classify_sale_udf(F.col("montant")))
categorized_df_UDF.show()

+-----------+---+----------+----------+------------+--------+-------------+------+--------+-------+----------+--------+----------------+-------+------------+
|customer_id| id|      date|   produit|   categorie|quantite|prix_unitaire|region|     nom| prenom|     ville| segment|date_inscription|montant|taille_vente|
+-----------+---+----------+----------+------------+--------+-------------+------+--------+-------+----------+--------+----------------+-------+------------+
|         15|  1|2024-01-04|    Souris| Accessoires|       5|         25.5|   Sud| Bernard|  Manon|     Lille| Premium|      2022-02-11|  127.5|       petit|
|         38|  2|2024-01-14|   Clavier| Accessoires|       9|         45.0|  Nord|  Durand|   Emma|     Lille|Standard|      2021-05-31|  405.0|       moyen|
|         14|  3|2024-01-05|    Casque| Accessoires|       1|         35.0|  Nord|  Moreau|Nicolas|      Lyon|     VIP|      2022-01-29|   35.0|       petit|
|         36|  4|2024-03-05|     Écran|Informatique|

#### Catégorisation avec natif Spark

In [28]:
%%time
categorized_df_native = df.withColumn(
    "taille_vente_native",
    F.when(F.col("montant") <= 210, "petit")
     .when(F.col("montant") < 750, "moyen")
     .otherwise("gros")
)
categorized_df_native.show()

+-----------+---+----------+----------+------------+--------+-------------+------+--------+-------+----------+--------+----------------+-------+-------------------+
|customer_id| id|      date|   produit|   categorie|quantite|prix_unitaire|region|     nom| prenom|     ville| segment|date_inscription|montant|taille_vente_native|
+-----------+---+----------+----------+------------+--------+-------------+------+--------+-------+----------+--------+----------------+-------+-------------------+
|         15|  1|2024-01-04|    Souris| Accessoires|       5|         25.5|   Sud| Bernard|  Manon|     Lille| Premium|      2022-02-11|  127.5|              petit|
|         38|  2|2024-01-14|   Clavier| Accessoires|       9|         45.0|  Nord|  Durand|   Emma|     Lille|Standard|      2021-05-31|  405.0|              moyen|
|         14|  3|2024-01-05|    Casque| Accessoires|       1|         35.0|  Nord|  Moreau|Nicolas|      Lyon|     VIP|      2022-01-29|   35.0|              petit|
|         

#### Résultats : UDF Python vs fonction native


| Version | Wall time |
|---|---|
| UDF Python | 1,14 s |
| Fonction native | 213 ms |

La version native est environ 5 fois plus rapide.

Pourquoi l'UDF est plus lente ?
- Spark tourne dans la JVM (Java), alors qu'une UDF est du code Python. Pour l'exécuter, Spark doit démarrer un processus Python à côté.
- Chaque ligne fait ensuite un aller-retour : Spark la convertit et l'envoie au processus Python, la fonction calcule la catégorie, puis le résultat est renvoyé à Spark et reconverti.

Pourquoi la version native est plus rapide ?
- F.when / otherwise est une fonction intégrée à Spark : tout le calcul reste dans la JVM, sans passage par Python.
- Spark connaît cette fonction et peut l'optimiser, par exemple en la compilant avec le reste du plan d'exécution.

## 3. Spark Streaming

```
-------------------------------------------                                     
Batch: 1
-------------------------------------------
+-------------------+-------------------+----------+--------+
|debut              |fin                |produit   |ca_total|
+-------------------+-------------------+----------+--------+
|2026-10-19 14:30:50|2026-10-19 14:31:00|Clavier   |495.0   |
|2026-10-19 14:30:40|2026-10-19 14:30:50|Casque    |210.0   |
|2026-10-19 14:30:40|2026-10-19 14:30:50|Souris    |178.5   |
|2026-10-19 14:30:50|2026-10-19 14:31:00|Ordinateur|3400.0  |
|2026-10-19 14:30:50|2026-10-19 14:31:00|Écran     |2420.0  |
|2026-10-19 14:30:50|2026-10-19 14:31:00|Souris    |306.0   |
+-------------------+-------------------+----------+--------+

-------------------------------------------                                     
Batch: 2
-------------------------------------------
+-------------------+-------------------+-------+--------+
|debut              |fin                |produit|ca_total|
+-------------------+-------------------+-------+--------+
|2026-10-19 14:30:50|2026-10-19 14:31:00|Clavier|900.0   |
|2026-10-19 14:30:50|2026-10-19 14:31:00|Écran  |2860.0  |
|2026-10-19 14:30:50|2026-10-19 14:31:00|Souris |739.5   |
|2026-10-19 14:30:50|2026-10-19 14:31:00|Casque |105.0   |
+-------------------+-------------------+-------+--------+

-------------------------------------------                                     
Batch: 3
-------------------------------------------
+-------------------+-------------------+-------+--------+
|debut              |fin                |produit|ca_total|
+-------------------+-------------------+-------+--------+
|2026-10-19 14:31:00|2026-10-19 14:31:10|Souris |153.0   |
|2026-10-19 14:31:00|2026-10-19 14:31:10|Écran  |1760.0  |
|2026-10-19 14:30:50|2026-10-19 14:31:00|Souris |1275.0  |
|2026-10-19 14:30:50|2026-10-19 14:31:00|Casque |350.0   |
+-------------------+-------------------+-------+--------+
```

### Observations Initiales 

Chaque ligne correspond au CA par produit sur une fenêtre de 10 secondes. Les colonnes Début / Fin sont les bornes temporelles de la fenètre 

Chaque batch affiche un tableau de 4 colonnes: 
1. Début de la fenêtre temporelle 
2. Fin de la fenêtre temporelle 
3. Nom du produit 
4. CA total réalisé durant la fenêtre temporelle


### readStream VS read



On obtient l'erreur :

`pyspark.errors.exceptions.captured.AnalysisException: socket is not a valid Spark SQL Data Source.`

`spark.read` attend une source **bornée**, c'est-à-dire un ensemble de données avec un début et une fin, comme un fichier ou une table. Une `socket` est au contraire un flux **non borné** : les données arrivent en continu et il n'y a pas de fin de dataset. Spark ne propose donc la source `socket` qu'en streaming, et le programme échoue parce qu'on demande une lecture batch sur une source qui ne peut être lue qu'en flux.

### Fenêtre temporelle 

Les fenêtres sont fixes et ne se chevauchent pas (tumbling windows). On a donc 60 / 5 = 12 fenêtres par minute avec une taille de 5 secondes, contre 6 avec 10 secondes.

Avec une fenêtre de 1 seconde, chaque fenêtre contient presque toujours une seule vente, l’agrégation ne regroupe quasiment rien et perds sont intérêt: le résultat ressemble aux événements bruts. Chaque batch couvre plusieurs fenêtres.
Un micro-batch dure quelques secondes, donc il contient 5-8 fenêtres différentes. Par exemple, le batch 1 va de 19:55:59 à 19:56:04.


### Output modes 


#### Append mode 

Spark lève l'erreur suivante 

```
pyspark.errors.exceptions.captured.AnalysisException: [STREAMING_OUTPUT_MODE.UNSUPPORTED_OPERATION] Invalid streaming output mode: append. This output mode is not supported for streaming aggregations without watermark on streaming DataFrames/DataSets. SQLSTATE: 42KDE
```

En mode append, chaque ligne n’est écrite qu’une seule fois, et elle ne doit plus jamais changer ensuite. Or une fenêtre d’agrégation peut toujours recevoir un événement en retard qui modifie son ca_total. Sans watermark, Spark ne sait jamais quand une fenêtre est définitivement close. Il ne peut donc rien émettre, et il refuse la requête.


#### Complete mode 

```

-------------------------------------------                                     
Batch: 1
-------------------------------------------
+-------------------+-------------------+----------+--------+
|debut              |fin                |produit   |ca_total|
+-------------------+-------------------+----------+--------+
|2026-10-19 20:01:20|2026-10-19 20:01:30|Souris    |255.0   |
|2026-10-19 20:01:20|2026-10-19 20:01:30|Ordinateur|5100.0  |
|2026-10-19 20:01:30|2026-10-19 20:01:40|Clavier   |90.0    |
|2026-10-19 20:01:30|2026-10-19 20:01:40|Écran     |440.0   |
|2026-10-19 20:01:20|2026-10-19 20:01:30|Casque    |175.0   |
|2026-10-19 20:01:30|2026-10-19 20:01:40|Souris    |306.0   |
|2026-10-19 20:01:30|2026-10-19 20:01:40|Casque    |245.0   |
|2026-10-19 20:01:20|2026-10-19 20:01:30|Écran     |220.0   |
+-------------------+-------------------+----------+--------+

-------------------------------------------                                     
Batch: 2
-------------------------------------------
+-------------------+-------------------+----------+--------+
|debut              |fin                |produit   |ca_total|
+-------------------+-------------------+----------+--------+
|2026-10-19 20:01:20|2026-10-19 20:01:30|Souris    |255.0   |
|2026-10-19 20:01:30|2026-10-19 20:01:40|Ordinateur|13600.0 |
|2026-10-19 20:01:20|2026-10-19 20:01:30|Ordinateur|5100.0  |
|2026-10-19 20:01:30|2026-10-19 20:01:40|Clavier   |990.0   |
|2026-10-19 20:01:30|2026-10-19 20:01:40|Écran     |2860.0  |
|2026-10-19 20:01:20|2026-10-19 20:01:30|Casque    |175.0   |
|2026-10-19 20:01:30|2026-10-19 20:01:40|Souris    |306.0   |
|2026-10-19 20:01:30|2026-10-19 20:01:40|Casque    |245.0   |
|2026-10-19 20:01:20|2026-10-19 20:01:30|Écran     |220.0   |
+-------------------+-------------------+----------+--------+

-------------------------------------------                                     
Batch: 3
-------------------------------------------
+-------------------+-------------------+----------+--------+
|debut              |fin                |produit   |ca_total|
+-------------------+-------------------+----------+--------+
|2026-10-19 20:01:20|2026-10-19 20:01:30|Souris    |255.0   |
|2026-10-19 20:01:30|2026-10-19 20:01:40|Ordinateur|13600.0 |
|2026-10-19 20:01:20|2026-10-19 20:01:30|Ordinateur|5100.0  |
|2026-10-19 20:01:30|2026-10-19 20:01:40|Clavier   |1305.0  |
|2026-10-19 20:01:30|2026-10-19 20:01:40|Écran     |3520.0  |
|2026-10-19 20:01:20|2026-10-19 20:01:30|Casque    |175.0   |
|2026-10-19 20:01:30|2026-10-19 20:01:40|Souris    |561.0   |
|2026-10-19 20:01:30|2026-10-19 20:01:40|Casque    |245.0   |
|2026-10-19 20:01:20|2026-10-19 20:01:30|Écran     |220.0   |
+-------------------+-------------------+----------+--------+

-------------------------------------------                                     
Batch: 4
-------------------------------------------
+-------------------+-------------------+----------+--------+
|debut              |fin                |produit   |ca_total|
+-------------------+-------------------+----------+--------+
|2026-10-19 20:01:40|2026-10-19 20:01:50|Ordinateur|3400.0  |
|2026-10-19 20:01:20|2026-10-19 20:01:30|Souris    |255.0   |
|2026-10-19 20:01:30|2026-10-19 20:01:40|Ordinateur|20400.0 |
|2026-10-19 20:01:20|2026-10-19 20:01:30|Ordinateur|5100.0  |
|2026-10-19 20:01:30|2026-10-19 20:01:40|Clavier   |1305.0  |
|2026-10-19 20:01:40|2026-10-19 20:01:50|Clavier   |90.0    |
|2026-10-19 20:01:30|2026-10-19 20:01:40|Écran     |3520.0  |
|2026-10-19 20:01:20|2026-10-19 20:01:30|Casque    |175.0   |
|2026-10-19 20:01:30|2026-10-19 20:01:40|Souris    |637.5   |
|2026-10-19 20:01:30|2026-10-19 20:01:40|Casque    |560.0   |
|2026-10-19 20:01:20|2026-10-19 20:01:30|Écran     |220.0   |
+-------------------+-------------------+----------+--------+

```

* Toute la table est réaffichée à chaque batch: Les lignes de la fenêtre 20:01:20 → 20:01:30 (Souris 255.0, Ordinateur 5100.0, Casque 175.0, Écran 220.0) apparaissent dans tous les batchs, alors qu’elles ne changent plus depuis le batch 1. Avec update, elles n’auraient été affichées qu’une fois.

* Les valeurs s’accumulent au fil des batchs: La fenêtre 20:01:30 → 20:01:40 reçoit des ventes pendant plusieurs batchs, et son ca_total augmente.

### Tableau Output Modes

| Mode | Ce qui est écrit | Quand l'utiliser |
|---|---|---|
| `append` | Uniquement les **nouvelles lignes définitives**, chacune écrite une seule fois. | Quand chaque résultat est final et ne change plus (filtres, transformations sans agrégation). |
| `update` | Uniquement les lignes **nouvelles ou modifiées** depuis le batch précédent. | Pour suivre des agrégats en quasi temps réel sans tout réécrire, mettre à jour que ce qui change. |
| `complete` | **Toute la table de résultats** à chaque batch, y compris les lignes inchangées. | Pour de petits résultats dont on veut toujours la vue complète et triée, comme un tableau de bord. |

### Streaming.py modifié pour filtrer le Nord 

```python


# ---------- 3. Parsing du CSV ----------
# Format reçu : id,event_time,produit,quantite,prix_unitaire,region
sales = (lines
    .select(split(col("value"), ",").alias("c"))
    .select(
        col("c")[0].cast("int").alias("id"),
        col("c")[1].cast("timestamp").alias("event_time"),
        col("c")[2].alias("produit"),
        col("c")[3].cast("int").alias("quantite"),
        col("c")[4].cast("double").alias("prix_unitaire"),
        col("c")[5].alias("region"),
    )
    .withColumn("ca", col("quantite") * col("prix_unitaire"))
    .filter(col("region") == "Nord"))
    
```

## 4. Pipeline ML Complet avec MLlib

In [57]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml import PipelineModel

In [30]:
spark = SparkSession.builder.appName("TP-Diabetes").getOrCreate()

26/10/24 10:08:32 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


Chargement du DataSet et exploration des données


In [31]:
df = spark.read.csv('/Users/antoinechosson/Spark-lab/Spark-Introduction-Lab/TP4/diabetes_012_health_indicators_BRFSS2015.csv',
                     header=True, inferSchema=True)

df.head()

Row(Diabetes_012=0.0, HighBP=1.0, HighChol=1.0, CholCheck=1.0, BMI=40.0, Smoker=1.0, Stroke=0.0, HeartDiseaseorAttack=0.0, PhysActivity=0.0, Fruits=0.0, Veggies=1.0, HvyAlcoholConsump=0.0, AnyHealthcare=1.0, NoDocbcCost=0.0, GenHlth=5.0, MentHlth=18.0, PhysHlth=15.0, DiffWalk=1.0, Sex=0.0, Age=9.0, Education=4.0, Income=3.0)

In [32]:
(df.count(), len(df.columns))

(253680, 22)

In [33]:
total = df.count()

(df.groupBy("Diabetes_012")
   .count()
   .withColumn("proportion", col("count") / total)
   .orderBy("Diabetes_012")
   .show())

+------------+------+--------------------+
|Diabetes_012| count|          proportion|
+------------+------+--------------------+
|         0.0|213703|  0.8424116997792495|
|         1.0|  4631|0.018255282245348472|
|         2.0| 35346| 0.13933301797540207|
+------------+------+--------------------+



Le dataset contient 253680 lignes (patients) et 22 colonnes (attributs). Il n'est pas équilibré entre les classes 0:Sain, 1:Diabète Type1 et 2:Diabète Type2. La proportion de personnes en classe 1 est **1.83%**.

Séparation features - labels 


In [46]:
df = df.withColumn("label", F.when(F.col("Diabetes_012") > 0, 1.0).otherwise(0.0))

### Rôle de StringIndexer et OneHotEncoder

### Pourquoi encoder les colonnes catégorielles ?

Les algorithmes de ML ne travaillent qu’avec des nombres, regroupés dans un vecteur de features (VectorAssembler). Une colonne texte comme "Nord" ou "Homme" doit donc être convertie avant l’entraînement. L’encodage sert à préciser au modèle comment interpréter ces nombres : soit comme des catégories distinctes, soit comme des valeurs ordonnées.

### Que se passerait-il si on les utilisait directement comme nombres ?

Le modèle supposerait un ordre et des distances entre les catégories. Si on code par exemple Nord = 0, Sud = 1, Est = 2. Une régression logistique en conclurait la hiérarchie: Est meilleur que Sud meilleur que Nord. Il concluerait également que Est est deux fois plus éloigné de Nord que de sud. Ainsi, l'effet sur la cible augmenterait de façon linéaire d’une région à l’autre. Mais les codes sont arbitraires. 

### Différence entre StringIndexer et OneHotEncoder

* **StringIndexer**	: Transforme chaque catégorie en un indice numérique (0, 1, 2…), en attribuant 0 à la catégorie la plus fréquente. Problème d'ordre implicite. 

* **OneHotEncoder** : Transforme un indice en un vecteur binaire, avec un 1 à la position de la catégorie. Pas d'ordre implicite, chaque catégorie devient une dimension indépendante. 

In [47]:
binary_cols = ['HighBP', 'HighChol', 'CholCheck', 'Smoker', 'Stroke',
               'HeartDiseaseorAttack', 'PhysActivity', 'Fruits', 'Veggies',
               'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost',
               'DiffWalk', 'Sex']
ordinal_cols = ['GenHlth', 'Age', 'Education', 'Income']
num_cols = ['BMI', 'MentHlth', 'PhysHlth']

idx_cols = [c + "_idx" for c in ordinal_cols]
ohe_cols = [c + "_ohe" for c in ordinal_cols]

indexer = StringIndexer(inputCols=ordinal_cols, outputCols=idx_cols,
                        handleInvalid="keep")
encoder = OneHotEncoder(inputCols=idx_cols, outputCols=ohe_cols)

## Assemblage des features et normalisation

#### Rôle de l'assemblage 

Les estimateurs de MLlib prennent en entrée un seul vecteur featuresCol et non une liste de colonnes. De cette manière tous les modèles ont la même entrée, quel que soit le nombre ou le type des features. Chaque ligne devient un vecteur numérique, directement exploitable par les calculs d’algèbre linéaire distribués.

La colonne features est un vecteur de 49 dimensions. Il contient ces éléments concaténés: 

* les 14 variables binaires (0/1) telles que HighBP, HighChol etc 
* les vecteurs one-hot des 4 variables ordinales (chacun possède k−1 composantes, où k est le nombre de catégories)
* les 3 variables numériques (BMI, MentHlth, PhysHlth)


#### Rôle de StandardScaler 

Les variables ont des échelles très différentes. BMI va d’environ 12 à 98, MentHlth et PhysHlth vont de 0 à 30, et les binaires valent 0 ou 1. StandardScaler permer de ramener les features sur une échelle égale.

Quel serait l’impact sur l’entraînement sans scaling: Une variable à grande échelle aurait naturellement un petit coefficient et serait donc peu pénalisée, alors qu’une binaire serait sur-pénalisée. Les coefficients sont non comparables car appliqués à des valeurs d'échelles différentes.

In [48]:
feature_cols = binary_cols + ordinal_cols + num_cols

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features_raw",
                            handleInvalid="keep")
scaler = StandardScaler(inputCol="features_raw", outputCol="features",
                        withStd=True, withMean=False)

In [52]:
rf = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=100, maxDepth=10)
pipeline_rf = Pipeline(stages=[indexer, encoder, assembler, scaler, rf])

#### Entrainement

In [53]:
train, test = df.randomSplit([0.8, 0.2], seed=42)
model = pipeline_rf.fit(train)
preds = model.transform(test)

26/10/24 13:50:28 WARN DAGScheduler: Broadcasting large task binary with size 1003.3 KiB
26/10/24 13:50:29 WARN DAGScheduler: Broadcasting large task binary with size 1907.5 KiB
26/10/24 13:50:31 WARN DAGScheduler: Broadcasting large task binary with size 3.6 MiB
26/10/24 13:50:35 WARN DAGScheduler: Broadcasting large task binary with size 1073.8 KiB
26/10/24 13:50:35 WARN DAGScheduler: Broadcasting large task binary with size 6.9 MiB
26/10/24 13:50:38 WARN DAGScheduler: Broadcasting large task binary with size 1947.6 KiB
26/10/24 13:50:39 WARN DAGScheduler: Broadcasting large task binary with size 12.7 MiB
26/10/24 13:50:45 WARN DAGScheduler: Broadcasting large task binary with size 3.3 MiB


In [54]:
auc = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC").evaluate(preds)
mc = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction")

print(f"AUC                 {auc:.4f}")
for m in ["accuracy", "weightedPrecision", "weightedRecall", "f1"]:
    print(f"{m:20s}{mc.evaluate(preds, {mc.metricName: m}):.4f}")

26/10/24 13:50:48 WARN DAGScheduler: Broadcasting large task binary with size 6.1 MiB


AUC                 0.8177


26/10/24 13:50:50 WARN DAGScheduler: Broadcasting large task binary with size 6.1 MiB


accuracy            0.8531


26/10/24 13:50:52 WARN DAGScheduler: Broadcasting large task binary with size 6.1 MiB


weightedPrecision   0.8248


26/10/24 13:50:54 WARN DAGScheduler: Broadcasting large task binary with size 6.1 MiB


weightedRecall      0.8531


26/10/24 13:50:56 WARN DAGScheduler: Broadcasting large task binary with size 6.1 MiB


f1                  0.8103


### Évaluation du modèle 

| Métrique | Valeur | 
|---|---|
| AUC | 0.8177 |
| Accuracy | 0.8531 |
| Weighted Precision | 0.8248 |
| Weighted Recall | 0.8531 | 
| F1 | 0.8103 | 


L'AUC est la plus adaptée à un Dataset déséquilibré. L’accuracy ici est trompeuse car environ 84 % des individus sont non diabétiques. Un modèle qui prédit toujours « 0 » obtient déjà environ 84 % d’accuracy sans rien détecter. Les métriques pondérées sont aussi biaisées. Weighted precision, weighted recall et F1 pondèrent chaque classe par son effectif. La classe majoritaire domine donc le score et masque les mauvaises performances sur la classe diabétique.

L’AUC est insensible au déséquilibre. Elle mesure la capacité du modèle à classer un diabétique au-dessus d’un non-diabétique, indépendamment du seuil de décision et de la proportion des classes. 

### Pipeline VS étapes manuelles

* **Reproductibilité** : les étapes s'appliquent toujours dans le même ordre, que ce soit pour l'entraînement, le test ou de nouvelles données. Pas de risque d'oublier une étape ou de se tromper d'ordre.
* **Sauvegarde simplifiée** : un seul `model.save()` suffit pour enregistrer à la fois le prétraitement et le modèle. Sans pipeline, il faudrait sauvegarder chaque étape une par une.
* **Code plus simple** : un seul `.fit()` et un seul `.transform()` au lieu de créer plein de DataFrames intermédiaires. Pour ajouter ou retirer une étape, il suffit de modifier la liste `stages`.
* **Optimisation des hyperparamètres** : on peut passer le pipeline entier à un `CrossValidator` pour régler les paramètres de toutes les étapes en même temps.

### Sauvegarde et rechargement 

In [56]:
model.write().overwrite().save("./model_diabetes")

26/10/24 13:52:27 WARN TaskSetManager: Stage 256 contains a task of very large size (2153 KiB). The maximum recommended task size is 1000 KiB.


In [ ]:
# Inférer avec le modèle sur des nouvelles données nouvelles_donnees.csv

loaded_model = PipelineModel.load("./model_diabetes")

new_data_df = spark.read.csv("nouvelles_donnees.csv", header=True, inferSchema=True)
preds_new = loaded_model.transform(new_data_df)
preds_new.select("prediction", "probability").show(5)

#### Fichiers créés dans ./model_diabetes/

* **metadata** : un fichier JSON qui décrit le pipeline (type d'objet, paramètres, ordre des étapes).
* **stages** : un sous-dossier par étape du pipeline, numéroté dans l'ordre :
  * 0_StringIndexer: les correspondances catégorie → index apprises
  * 1_OneHotEncoder: le nombre de catégories par colonne
  * 2_VectorAssembler:  seulement les paramètres, car rien n'est appris à cette étape
  * 3_StandardScaler: les écarts-types calculés sur le train
  * 4_RandomForestClassifier: la structure des 100 arbres (nœuds, seuils, prédictions)

Chaque étape a un dossier metadata (paramètres en JSON). Celles qui ont appris quelque chose ont aussi un dossier data qui stocke ces valeurs au format Parquet. Des fichiers SUCCESS confirment que l'écriture s'est bien terminée.